# Panzer — Endpoints autenticados (trading)

Este notebook muestra como usar `BinanceClient` para operar con endpoints
que requieren firma HMAC-SHA256.

Las respuestas se muestran tal cual las devuelve la API de Binance — son
exactamente lo que devuelve panzer, sin transformaciones.

**Requisitos:**
- API key y API secret de Binance.
- La primera vez Panzer te las pide por terminal y las almacena cifradas
  en `~/.panzer_creds`. Las siguientes veces las carga automaticamente.

> **ATENCION:** Los ejemplos de ordenes estan comentados para evitar
> operaciones accidentales con dinero real.

## 1. Crear el cliente

In [1]:
from panzer import BinanceClient

client = BinanceClient(market="spot")

2026-03-06 13:16:17     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-06 13:16:18     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90
2026-03-06 13:16:18     INFO [panzer.binance.client.spot] BinanceClient inicializado (market=spot)


## 2. account() — respuesta cruda

La respuesta de `/api/v3/account` es un dict grande. Mostramos las claves
de primer nivel y luego los balances con saldo.

In [2]:
account = client.account()

# Claves que trae la respuesta
print("Claves de la respuesta account():")
print(list(account.keys()))

Claves de la respuesta account():
['makerCommission', 'takerCommission', 'buyerCommission', 'sellerCommission', 'commissionRates', 'canTrade', 'canWithdraw', 'canDeposit', 'brokered', 'requireSelfTradePrevention', 'preventSor', 'updateTime', 'accountType', 'balances', 'permissions', 'uid']


In [3]:
# Campos escalares de la respuesta (todo menos balances y permissions)
{k: v for k, v in account.items() if k not in ("balances", "permissions")}

{'makerCommission': 10,
 'takerCommission': 10,
 'buyerCommission': 0,
 'sellerCommission': 0,
 'commissionRates': {'maker': '0.00100000',
  'taker': '0.00100000',
  'buyer': '0.00000000',
  'seller': '0.00000000'},
 'canTrade': True,
 'canWithdraw': True,
 'canDeposit': True,
 'brokered': False,
 'requireSelfTradePrevention': False,
 'preventSor': False,
 'updateTime': 1772783102659,
 'accountType': 'SPOT',
 'uid': 37237719}

In [4]:
# Balances con saldo > 0 — cada elemento tal cual viene de Binance
[b for b in account["balances"] if float(b["free"]) > 0 or float(b["locked"]) > 0]

[{'asset': 'USDT', 'free': '0.02055143', 'locked': '0.00000000'},
 {'asset': 'LDBTC', 'free': '0.06002833', 'locked': '0.00000000'},
 {'asset': 'LDETH', 'free': '1.26322149', 'locked': '0.00000000'},
 {'asset': 'LDSOL', 'free': '59.39125897', 'locked': '0.00000000'},
 {'asset': 'LDPEPE', 'free': '714.20', 'locked': '0.00'},
 {'asset': 'LDBONK', 'free': '1035.46', 'locked': '0.00'},
 {'asset': 'LDPIXEL', 'free': '28.14493831', 'locked': '0.00000000'},
 {'asset': 'LDW', 'free': '18.37203062', 'locked': '0.00000000'},
 {'asset': 'LDSXT', 'free': '15.57013617', 'locked': '0.00000000'}]

## 3. my_trades() — respuesta cruda

Cada trade es un dict con la estructura nativa de Binance.

In [5]:
my_trades = client.my_trades("BTCUSDT", limit=3)
my_trades

[{'symbol': 'BTCUSDT',
  'id': 3316660873,
  'orderId': 23687231478,
  'orderListId': -1,
  'price': '40986.19000000',
  'qty': '0.00128000',
  'quoteQty': '52.46232320',
  'commission': '0.00016332',
  'commissionAsset': 'BNB',
  'time': 1702320683140,
  'isBuyer': True,
  'isMaker': True,
  'isBestMatch': True},
 {'symbol': 'BTCUSDT',
  'id': 3327971249,
  'orderId': 23818929439,
  'orderListId': -1,
  'price': '43012.60000000',
  'qty': '0.00303000',
  'quoteQty': '130.32817800',
  'commission': '0.00039912',
  'commissionAsset': 'BNB',
  'time': 1702981648395,
  'isBuyer': False,
  'isMaker': False,
  'isBestMatch': True},
 {'symbol': 'BTCUSDT',
  'id': 4253645893,
  'orderId': 33659658457,
  'orderListId': 3401119533,
  'price': '100494.06000000',
  'qty': '0.01725000',
  'quoteQty': '1733.52253500',
  'commission': '1.73352254',
  'commissionAsset': 'USDT',
  'time': 1733935275226,
  'isBuyer': False,
  'isMaker': False,
  'isBestMatch': True}]

## 4. open_orders() — respuesta cruda

In [6]:
# Lista vacia si no hay ordenes abiertas
open_orders = client.open_orders("BTCUSDT")
open_orders

[]

## 5. all_orders() — respuesta cruda

Historial completo: abiertas, cerradas, canceladas. Cada orden es un dict
con todos los campos que devuelve Binance.

In [7]:
all_orders = client.all_orders("BTCUSDT", limit=3)
all_orders

[{'symbol': 'BTCUSDT',
  'orderId': 23687231478,
  'orderListId': -1,
  'clientOrderId': 'ios_59e8a6cb507043979eb9c96f3a3ea9d2',
  'price': '40986.19000000',
  'origQty': '0.00128000',
  'executedQty': '0.00128000',
  'cummulativeQuoteQty': '52.46232320',
  'status': 'FILLED',
  'timeInForce': 'GTC',
  'type': 'LIMIT',
  'side': 'BUY',
  'stopPrice': '0.00000000',
  'icebergQty': '0.00000000',
  'time': 1702320674170,
  'updateTime': 1702320683140,
  'isWorking': True,
  'workingTime': 1702320674170,
  'origQuoteOrderQty': '0.00000000',
  'selfTradePreventionMode': 'EXPIRE_MAKER'},
 {'symbol': 'BTCUSDT',
  'orderId': 23818929439,
  'orderListId': -1,
  'clientOrderId': 'web_729a25d1aa1a4edfb5f202cb7302c9fe',
  'price': '43010.68000000',
  'origQty': '0.00303000',
  'executedQty': '0.00303000',
  'cummulativeQuoteQty': '130.32817800',
  'status': 'FILLED',
  'timeInForce': 'GTC',
  'type': 'LIMIT',
  'side': 'SELL',
  'stopPrice': '0.00000000',
  'icebergQty': '0.00000000',
  'time': 17

## 6. Crear ordenes

> **CUIDADO:** Estas celdas crean ordenes REALES. Estan comentadas por seguridad.

### Orden LIMIT — respuesta esperada

```python
{
    "symbol": "BTCUSDT",
    "orderId": 28,
    "orderListId": -1,
    "clientOrderId": "6gCrw2kRUAF9CvJDGP16IP",
    "transactTime": 1507725176595,
    "price": "50000.00000000",
    "origQty": "0.00100000",
    "executedQty": "0.00000000",
    "cummulativeQuoteQty": "0.00000000",
    "status": "NEW",
    "timeInForce": "GTC",
    "type": "LIMIT",
    "side": "BUY",
    "workingTime": 1507725176595,
    "fills": [],
    "selfTradePreventionMode": "EXPIRE_TAKER"
}
```

In [8]:
# DESCOMENTA PARA EJECUTAR:

# order = client.new_order(
#     symbol="BTCUSDT",
#     side="BUY",
#     order_type="LIMIT",
#     quantity=0.001,
#     price=50000.0,
#     time_in_force="GTC",
# )
# order  # respuesta cruda de Binance

### Orden MARKET — respuesta esperada

Las ordenes MARKET se ejecutan inmediatamente. La respuesta incluye `fills`
con los precios y cantidades reales de ejecucion.

```python
{
    "symbol": "BTCUSDT",
    "orderId": 29,
    "status": "FILLED",
    "type": "MARKET",
    "side": "BUY",
    "executedQty": "0.00010000",
    "cummulativeQuoteQty": "9.12340000",
    "fills": [
        {
            "price": "91234.00000000",
            "qty": "0.00010000",
            "commission": "0.00000010",
            "commissionAsset": "BTC",
            "tradeId": 123456
        }
    ]
}
```

In [9]:
# DESCOMENTA PARA EJECUTAR:

# market_order = client.new_order(
#     symbol="BTCUSDT",
#     side="BUY",
#     order_type="MARKET",
#     quote_order_qty=10.0,   # comprar 10 USDT de BTC
# )
# market_order  # respuesta cruda

### Cancelar una orden

In [10]:
# DESCOMENTA PARA EJECUTAR:

# cancelled = client.cancel_order("BTCUSDT", order_id=12345678)
# cancelled  # respuesta cruda

## 7. signed_request() — endpoint generico

Para cualquier endpoint autenticado sin wrapper dedicado, usa
`signed_request()`. Panzer anade timestamp, firma y API key automaticamente.

In [11]:
# Ejemplo: obtener depositAddress (requiere firma completa)
# data = client.signed_request(
#     "GET",
#     "/sapi/v1/capital/deposit/address",
#     params=[("coin", "BTC"), ("network", "BTC")],
# )
# data  # respuesta cruda

# Ejemplo: listen key (semi-firmado, solo API key, sin HMAC)
# listen = client.signed_request(
#     "POST",
#     "/api/v3/userDataStream",
#     sign=False,
# )
# listen  # {"listenKey": "pqia91ma19a5s61cv6a81va65sdf19v8a65a1a5s61cv6a81va65sdf19v8a65a1"}

## 8. Futuros (UM / CM)

Los mismos metodos funcionan con futuros. Solo cambia el `market` al crear
el cliente. Panzer resuelve automaticamente los endpoints correctos.

```python
# USDT-M Futures
um_client = BinanceClient(market="um")
um_account = um_client.account()           # /fapi/v2/account
um_trades = um_client.my_trades("BTCUSDT") # /fapi/v1/userTrades
um_orders = um_client.all_orders("BTCUSDT")# /fapi/v1/allOrders

# COIN-M Futures
cm_client = BinanceClient(market="cm")
cm_account = cm_client.account()                # /dapi/v1/account
cm_trades = cm_client.my_trades("BTCUSD_PERP")  # /dapi/v1/userTrades
```

Los endpoints privados por mercado:

| Metodo | spot | um | cm |
|---|---|---|---|
| `account()` | /api/v3/account | /fapi/v2/account | /dapi/v1/account |
| `my_trades()` | /api/v3/myTrades | /fapi/v1/userTrades | /dapi/v1/userTrades |
| `new_order()` | /api/v3/order | /fapi/v1/order | /dapi/v1/order |
| `open_orders()` | /api/v3/openOrders | /fapi/v1/openOrders | /dapi/v1/openOrders |
| `all_orders()` | /api/v3/allOrders | /fapi/v1/allOrders | /dapi/v1/allOrders |

## 9. Manejo de errores

Panzer levanta `BinanceAPIException` con el codigo y mensaje originales
de Binance. No transforma ni oculta nada.

```python
from panzer.errors import BinanceAPIException

try:
    client.my_trades("SIMBOLO_INVALIDO")
except BinanceAPIException as e:
    print(e.status_code)       # 400
    print(e.error_payload.code) # -1121
    print(e.error_payload.msg)  # "Invalid symbol."
    print(e.body)              # JSON crudo de la respuesta
```

Codigos de error comunes de Binance:

| Codigo | Significado |
|---|---|
| -1121 | Invalid symbol |
| -1021 | Timestamp fuera de recvWindow |
| -2010 | Insufficient balance |
| -2013 | Order does not exist |
| -1003 | Too many requests (rate limit) |